# Episode 5 — Button interrupts with asyncio

In episode 3 Python **polled** the buttons: it read them again and again in a loop.
That keeps a CPU core busy even when nothing happens.

With **interrupts**, the button block in the PL signals the processor when a button changes.
Python sleeps until then. PYNQ connects interrupts to Python's `asyncio`:

```python
await base.buttons[0].wait_for_value_async(1)   # sleeps until button 0 is pressed
```

In a notebook you can use `await` directly in a cell.

In [ ]:
from pynq.overlays.base import BaseOverlay
import asyncio, random, time

base = BaseOverlay("base.bit")
seg = base.seven_seg

## 1. Polling versus waiting: how busy is the CPU?

Each cell runs for 5 seconds. Don't press anything.

In [ ]:
cpu0, t0 = time.process_time(), time.time()
while time.time() - t0 < 5:
    base.buttons.read()                      # polling, no pause
print("Polling:   CPU busy {:.0f} % of the time".format(
    100 * (time.process_time() - cpu0) / (time.time() - t0)))

In [ ]:
cpu0, t0 = time.process_time(), time.time()
try:
    await asyncio.wait_for(base.buttons[0].wait_for_value_async(1), timeout=5)
except asyncio.TimeoutError:
    pass
print("Interrupt: CPU busy {:.0f} % of the time".format(
    100 * (time.process_time() - cpu0) / (time.time() - t0)))

## 2. Reaction-time game

When the LEDs light and the buzzer sounds, press **button 0** as fast as you can.
Pressing too early shows `SOON`. Five rounds.

Two things wait at the same time in each round: a random delay, and an early press.
`asyncio.wait()` waits for whichever comes first.

In [ ]:
async def one_round():
    base.leds[0:4].off()
    await base.buttons[0].wait_for_value_async(0)            # button released

    early = asyncio.ensure_future(base.buttons[0].wait_for_value_async(1))
    done, _ = await asyncio.wait([early], timeout=random.uniform(1.0, 3.0))
    if done:                                                  # pressed during the delay
        return None
    early.cancel()

    base.leds[0:4].write(0b1111)
    base.buzzer.tone(2000)
    t0 = time.perf_counter()
    await base.buttons[0].wait_for_value_async(1)
    ms = (time.perf_counter() - t0) * 1000
    base.buzzer.off()
    base.leds[0:4].off()
    return ms

times = []
for n in range(5):
    seg.show_text("r" + str(n + 1))
    await asyncio.sleep(1)
    ms = await one_round()
    if ms is None:
        seg.show_text("SOON")
        print("Round", n + 1, ": too soon!")
    else:
        times.append(ms)
        seg.show_number(int(ms))
        print("Round", n + 1, ": {:.0f} ms".format(ms))
    await asyncio.sleep(1.5)

if times:
    print("Best {:.0f} ms, average {:.0f} ms".format(min(times), sum(times) / len(times)))
    seg.show_number(int(min(times)))

## 3. Two players

Player 1 has button 0, player 2 has button 1. The first press after the beep wins.
`return_when=FIRST_COMPLETED` stops waiting as soon as one button is pressed.

In [ ]:
async def press(button):
    await base.buttons[button].wait_for_value_async(1)
    return button

seg.show_text("----")
await asyncio.sleep(random.uniform(2.0, 4.0))
base.buzzer.tone(1500)
tasks = [asyncio.ensure_future(press(0)), asyncio.ensure_future(press(1))]
done, pending = await asyncio.wait(tasks, return_when=asyncio.FIRST_COMPLETED)
base.buzzer.off()
for t in pending:
    t.cancel()

winners = sorted(t.result() for t in done)
if len(winners) > 1:                     # both pressed within the same interrupt
    seg.show_text("tIE")
    print("A tie!")
else:
    winner = winners[0]
    seg.show_text("P" + str(winner + 1))
    base.leds[0:4].write(0b0011 if winner == 0 else 0b1100)
    print("Player", winner + 1, "wins")

## Try it yourself

Run two tasks together with `asyncio.gather()`: one blinks LED 3 every 0.5 s,
the other waits for switch 0 to change and then cancels the blinking.

In [ ]:
# your code here